# MinIO Delta Lake Access with Spark
This notebook demonstrates how to connect to the MinIO Delta Lake tables using PySpark.

In [1]:
import os
from pyspark.sql import SparkSession

# Set up the SparkSession with Delta Lake and MinIO (S3A) configurations
spark = (
    SparkSession.builder.appName("Jupyter-Delta")
    # Note: 3.5.0 packages for Spark 3.5.0
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # MinIO connectivity
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session established!")

Spark session established!


In [9]:
! pip install minio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 6.2 MB/s eta 0:00:0000:0100:01


In [12]:
# # List all buckets in MinIO
# fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
# status = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path("s3a:///"))
# for fileStatus in status:
#     print(fileStatus.getPath())


from minio import Minio

client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin123",
    secure=False,
)

for bucket in client.list_buckets():
    print(bucket.name)

bronze
delta-tables
gold
landing
silver
warehouse


In [15]:
# Example of reading a Delta table (uncomment and replace with your actual bucket/table path)
branches_df = spark.read.format("delta").load("s3a://bronze/stg_reference_branches")
branches_df.show()

# Alternatively, if using Spark SQL:
# spark.sql("SHOW DATABASES").show()

+---------+--------------------+-------------+--------+-----------+---------+
|branch_id|         branch_name|     province|  region|opened_date|is_active|
+---------+--------------------+-------------+--------+-----------+---------+
|   BR0001|    Koshi Branch 001|        Koshi|REGION_2| 2011-02-02|     true|
|   BR0002|  Lumbini Branch 002|      Lumbini|REGION_3| 2012-03-03|     true|
|   BR0003|  Karnali Branch 003|      Karnali|REGION_4| 2013-04-04|     true|
|   BR0004|Sudurpashchim Bra...|Sudurpashchim|REGION_5| 2014-05-05|     true|
|   BR0005|  Karnali Branch 005|      Karnali|REGION_1| 2015-06-06|     true|
|   BR0006|  Lumbini Branch 006|      Lumbini|REGION_2| 2016-07-07|     true|
|   BR0007|  Madhesh Branch 007|      Madhesh|REGION_3| 2017-08-08|     true|
|   BR0008|  Bagmati Branch 008|      Bagmati|REGION_4| 2018-09-09|     true|
|   BR0009|  Karnali Branch 009|      Karnali|REGION_5| 2019-10-10|     true|
|   BR0010|  Bagmati Branch 010|      Bagmati|REGION_1| 2020-11-

In [16]:
#number of rows in the dataframe
print(f"Number of rows in the dataframe: {branches_df.count()}")

Number of rows in the dataframe: 40


In [2]:
#read table at landing/accounts/account_snapshot_2026-01-01/2026-07-26_05-12-07
account_snapshot_df = spark.read.format("delta").load("s3a://landing/accounts/account_snapshot_2026-01-01/2026-07-26_05-12-07")
account_snapshot_df.show()

+-------------+------------+---------+------------+--------------+-----------+--------+-------------+
|   account_id| customer_id|branch_id|account_type|account_status|opened_date|currency|snapshot_date|
+-------------+------------+---------+------------+--------------+-----------+--------+-------------+
|ACC0000000001|CUST00003422|   BR0034|      WALLET|        ACTIVE| 2023-03-30|     NPR|   2026-01-01|
|ACC0000000002|CUST00004583|   BR0018|      WALLET|        ACTIVE| 2024-10-03|     NPR|   2026-01-01|
|ACC0000000003|CUST00004720|   BR0007|      SALARY|        ACTIVE| 2021-11-09|     NPR|   2026-01-01|
|ACC0000000004|CUST00002130|   BR0015|      WALLET|        ACTIVE| 2023-04-17|     NPR|   2026-01-01|
|ACC0000000005|CUST00002667|   BR0005|     SAVINGS|        ACTIVE| 2024-04-04|     NPR|   2026-01-01|
|ACC0000000006|CUST00002592|   BR0031|        LOAN|        ACTIVE| 2023-12-03|     NPR|   2026-01-01|
|ACC0000000007|CUST00002914|   BR0015|     SAVINGS|        ACTIVE| 2024-05-24|    

In [19]:
#unique values including nulls in the column "account_type", "account status" and "currency" in the dataframe and the count of each unique value in the column
print("Unique values in the column 'account_type' and their counts:")
account_snapshot_df.groupBy("account_type").count().show()
print("Unique values in the column 'account_status' and their counts:")
account_snapshot_df.groupBy("account_status").count().show()
print("Unique values in the column 'currency' and their counts:")
account_snapshot_df.groupBy("currency").count().show()

Unique values in the column 'account_type' and their counts:
+------------+-----+
|account_type|count|
+------------+-----+
|      SALARY| 1606|
|        LOAN| 1567|
|     SAVINGS| 1648|
|      WALLET| 1576|
|     CURRENT| 1603|
+------------+-----+

Unique values in the column 'account_status' and their counts:
+--------------+-----+
|account_status|count|
+--------------+-----+
|       BLOCKED|  163|
|       DORMANT|  609|
|        CLOSED|  232|
|        ACTIVE| 6996|
+--------------+-----+

Unique values in the column 'currency' and their counts:
+--------+-----+
|currency|count|
+--------+-----+
|     NPR| 8000|
+--------+-----+



In [3]:
#read table at landing/customers/customer_snapshot_2026-01-01/2026-07-26_05-12-08/
customer_snapshot_df = spark.read.format("delta").load("s3a://landing/customers/customer_snapshot_2026-01-01/2026-07-26_05-12-08")
customer_snapshot_df.show()

+------------+---------------+-------+--------+-------------+---------+------------+-------------+---------+
| customer_id|  customer_name|segment|age_band|     province|risk_band|created_date|snapshot_date|is_active|
+------------+---------------+-------+--------+-------------+---------+------------+-------------+---------+
|CUST00000001|Subash Shrestha|PREMIUM|   18-24|        Koshi|   MEDIUM|  2025-08-11|   2026-01-01|     true|
|CUST00000002|    Maya Tamang|PREMIUM|   45-54|      Lumbini|     HIGH|  2019-02-09|   2026-01-01|     true|
|CUST00000003|     Puja Bista| RETAIL|   35-44|      Madhesh|      LOW|  2020-06-18|   2026-01-01|     true|
|CUST00000004|     Rita Bista|PREMIUM|   25-34|      Gandaki|   MEDIUM|  2020-04-23|   2026-01-01|     true|
|CUST00000005|    Anika Karki| RETAIL|   35-44|      Bagmati|      LOW|  2025-08-10|   2026-01-01|     true|
|CUST00000006|   Anika Poudel|STUDENT|     55+|        Koshi|     HIGH|  2020-07-14|   2026-01-01|     true|
|CUST00000007|    S

In [4]:
#unique values including nulls in the column "segment", "province", "risk_band", "is_active" in the dataframe and the count of each unique value in the column
for col in ["segment", "province", "risk_band", "is_active"]:
    print(f"Unique values in the column '{col}' and their counts:")
    customer_snapshot_df.groupBy(col).count().show()

Unique values in the column 'segment' and their counts:
+-------+-----+
|segment|count|
+-------+-----+
|STUDENT|  978|
|    SME| 1000|
|PREMIUM| 1007|
| RETAIL| 1020|
| SENIOR|  995|
+-------+-----+

Unique values in the column 'province' and their counts:
+-------------+-----+
|     province|count|
+-------------+-----+
|      Madhesh|  700|
|Sudurpashchim|  742|
|      Lumbini|  699|
|      Bagmati|  727|
|      Karnali|  690|
|      Gandaki|  726|
|        Koshi|  716|
+-------------+-----+

Unique values in the column 'risk_band' and their counts:
+---------+-----+
|risk_band|count|
+---------+-----+
|     HIGH| 1275|
|      LOW| 2486|
|   MEDIUM| 1239|
+---------+-----+

Unique values in the column 'is_active' and their counts:
+---------+-----+
|is_active|count|
+---------+-----+
|     true| 3756|
|    false| 1244|
+---------+-----+



In [ ]:
account_customer_ids = account_snapshot_df.select("customer_id").distinct()
customer_customer_ids = customer_snapshot_df.select("customer_id").distinct()

#check foreign key integrity between account_snapshot_df and customer_snapshot_df on customer_id
missing_customer_ids = account_customer_ids.subtract(customer_customer_ids)
if missing_customer_ids.count() > 0:
    print("Foreign key integrity check failed. Missing customer_ids in customer_snapshot_df:")
    missing_customer_ids.show()
else:
    print("Foreign key integrity check passed. All customer_ids in account_snapshot_df exist in customer_snapshot_df.")

Foreign key integrity check passed. All customer_ids in account_snapshot_df exist in customer_snapshot_df.
